# Reformat SY Export

### Step 0: Set-Up
Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [1]:
from typing import Tuple

import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm  # Progress bar

import climakitae as ck
from climakitae.explore.standard_year_profile import (
    get_climate_profile,
    export_profile_to_csv,
    set_profile_metadata,
    _format_based_on_structure,
    _construct_profile_dataframe,
    _create_simple_dataframe,
    _create_single_wl_multi_sim_dataframe,
    _create_multi_wl_single_sim_dataframe,
    _create_multi_wl_multi_sim_dataframe,
     _stack_profile_data
)
from climakitae.explore.typical_meteorological_year import TMY
from climakitae.core.data_interface import (
    get_data_options,
    get_subsetting_options,
    get_data,
)

import warnings
warnings.filterwarnings("ignore")


from unittest.mock import MagicMock, call, patch
import pytest

## Where is the issue?

In [2]:
time_delta = pd.date_range(
    "2020-01-01", periods=8760, freq="h"
)  # 1 year of hourly data
warming_levels = [1.5]
warming_levels_2 = [1.5, 2.0]
simulations = ["sim1"]
simulations_2 = ["sim1","sim2"]

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels), len(time_delta), len(simulations))

simple_data = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels,
        "time_delta": time_delta,
        "simulation": simulations,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels_2), len(time_delta), len(simulations))
multi_wl_data = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels_2,
        "time_delta": time_delta,
        "simulation": simulations,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels), len(time_delta), len(simulations_2))
multi_sim_data = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels,
        "time_delta": time_delta,
        "simulation": simulations_2,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels_2), len(time_delta), len(simulations_2))
multi_both_data = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels_2,
        "time_delta": time_delta,
        "simulation": simulations_2,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)


### Exploded final function

Alright, I see it now. Let's go from the top.

In [3]:
def _stack_profile_data(
    profile_data: dict,
    wl_names: list,
    sim_names: list,
) -> np.ndarray:
    """
    Stack profile data into a single array for DataFrame construction.

    Parameters
    ----------
    profile_data : dict
        Dictionary with (wl_name, sim_name) keys and profile arrays as values
    wl_names : list
        List of warming level names
    sim_names : list
        List of simulation names

    Returns
    -------
    np.ndarray
        Stacked data array ready for DataFrame construction
    """
    all_data = []

    for wl_name in wl_names:
        for sim_name in sim_names:
            key = (wl_name, sim_name)
            all_data.append(profile_data[key])

    return np.column_stack(all_data)

In [4]:
def _create_simple_dataframe(
    profile_data: dict,
    warming_level: float,
    simulation: any,
    sim_label_func: callable,
    hours_per_year: int,
) -> pd.DataFrame:
    """
    Create a simple DataFrame for single warming level and single simulation.

    Parameters
    ----------
    profile_data : dict
        Profile data dictionary
    warming_level : float
        Single warming level value
    simulation : any
        Single simulation identifier
    sim_label_func : callable
        Function to get simulation label
    hours_per_year : int
        Hours per year (8760)

    Returns
    -------
    pd.DataFrame
        Simple DataFrame with a single simulation column
    """

    wl_key = warming_level
    sim_key = sim_label_func(simulation, 0)

    # Stack data
    all_data = _stack_profile_data(
            profile_data=profile_data,
            wl_names=[f"WL_{wl_key}"],
            sim_names=[sim_key]
        )

    return  pd.DataFrame(
            all_data,
            columns=[sim_key],
            index=np.arange(1, hours_per_year + 1, 1),
        )


def _create_single_wl_multi_sim_dataframe(
    profile_data: dict,
    warming_level: float,
    simulations: list,
    sim_label_func: callable,
    hours_per_year: int,
) -> pd.DataFrame:
    """
    Create DataFrame for single warming level with multiple simulations.

    Parameters
    ----------
    profile_data : dict
        Profile data dictionary
    warming_level : float
        Single warming level value
    simulations : list
        List of simulation identifiers
    sim_label_func : callable
        Function to get simulation labels
    hours_per_year : int
        Hours per year (8760)

    Returns
    -------
    pd.DataFrame
        DataFrame with Simulation columns
    """
    wl = warming_level
    sim_names = [sim_label_func(sim, i) for i, sim in enumerate(simulations)]

    # Ensure simulation names are unique
    if len(sim_names) != len(set(sim_names)):
        print(
            "   ⚠️  Warning: Duplicate simulation names detected, adding uniqueness suffixes"
        )
        unique_sim_names = []
        name_counts = {}
        for name in sim_names:
            if name not in name_counts:
                name_counts[name] = 0
                unique_sim_names.append(name)
            else:
                name_counts[name] += 1
                unique_sim_names.append(f"{name}_v{name_counts[name]}")
        sim_names = unique_sim_names

    # Stack data
    all_data = _stack_profile_data(
        profile_data=profile_data,
        wl_names=[f"WL_{wl}"],
        sim_names=sim_names,
    )

    return pd.DataFrame(
        all_data,
        columns=simulations,
        index=np.arange(1, hours_per_year + 1, 1),
    )


def _create_multi_wl_single_sim_dataframe(
    profile_data: dict,
    warming_levels: np.ndarray,
    simulation: any,
    sim_label_func: callable,
    hours_per_year: int,
) -> pd.DataFrame:
    """
    Create DataFrame for multiple warming levels with single simulation.

    Parameters
    ----------
    profile_data : dict
        Profile data dictionary
    warming_levels : np.ndarray
        Array of warming level values
    simulation : any
        Single simulation identifier
    sim_label_func : callable
        Function to get simulation label
    hours_per_year : int
        Hours per year (8760)

    Returns
    -------
    pd.DataFrame
        DataFrame with Warming_Level columns
    """

    sim_name = sim_label_func(simulation, 0)
    wl_names = [f"WL_{wl}" for wl in warming_levels]

    # Create MultiIndex columns
    col_tuples = [(wl_name, sim) for wl_name in wl_names for sim in simulations]
    multi_cols = pd.MultiIndex.from_tuples(
        col_tuples, names=["Warming_Level", "Simulation"]
    )

    # Stack data
    all_data = _stack_profile_data(
        profile_data=profile_data,
        wl_names=wl_names,
        sim_names=[sim_name],
    )

    return pd.DataFrame(
        all_data,
        columns=multi_cols,
        index=np.arange(1, hours_per_year + 1, 1),
    )


def _create_multi_wl_multi_sim_dataframe(
    profile_data: dict,
    warming_levels: np.ndarray,
    simulations: list,
    sim_label_func: callable,
    hours_per_year: int,
) -> pd.DataFrame:
    """
    Create DataFrame for multiple warming levels and multiple simulations.

    Parameters
    ----------
    profile_data : dict
        Profile data dictionary
    warming_levels : np.ndarray
        Array of warming level values
    simulations : list
        List of simulation identifiers
    sim_label_func : callable
        Function to get simulation labels
    hours_per_year : int
        Hours per year (8760)

    Returns
    -------
    pd.DataFrame
        DataFrame with (Warming_Level, Simulation) MultiIndex columns
    """

    wl_names = [f"WL_{wl}" for wl in warming_levels]
    sim_names = [sim_label_func(sim, i) for i, sim in enumerate(simulations)]

    # Ensure simulation names are unique
    if len(sim_names) != len(set(sim_names)):
        print(
            "   ⚠️  Warning: Duplicate simulation names detected, adding uniqueness suffixes"
        )
        unique_sim_names = []
        name_counts = {}
        for name in sim_names:
            if name not in name_counts:
                name_counts[name] = 0
                unique_sim_names.append(name)
            else:
                name_counts[name] += 1
                unique_sim_names.append(f"{name}_v{name_counts[name]}")
        sim_names = unique_sim_names

    # Create MultiIndex columns
    col_tuples = [(wl_name, sim_name) for wl_name in wl_names for sim_name in sim_names]
    multi_cols = pd.MultiIndex.from_tuples(
        col_tuples, names=["Warming_Level", "Simulation"]
    )

    # Stack data
    all_data = _stack_profile_data(
        profile_data=profile_data,
        wl_names=wl_names,
        sim_names=sim_names,
    )

    return pd.DataFrame(
        all_data,
        columns=multi_cols,
        index=np.arange(1, hours_per_year + 1, 1),
    )

In [5]:
def _construct_profile_dataframe(
    profile_data: dict,
    warming_levels: np.ndarray,
    simulations: list,
    sim_label_func: callable,
    hours_per_year: int,
) -> pd.DataFrame:
    """
    Construct a DataFrame from profile data based on warming level and simulation dimensions.

    Parameters
    ----------
    profile_data : dict
        Dictionary with (warming_level, simulation) keys and profile arrays as values
    warming_levels : np.ndarray
        Array of warming level values
    simulations : list
        List of simulation identifiers
    sim_label_func : callable
        Function to extract simulation labels
    hours_per_year : int
        Hours per year (8760)

    Returns
    -------
    pd.DataFrame
        Structured DataFrame with appropriate column structure based on dimensions
    """

    n_warming_levels = len(warming_levels)
    n_simulations = len(simulations)

    if n_warming_levels == 1 and n_simulations == 1:
        return _create_simple_dataframe(
            profile_data,
            warming_levels[0],
            simulations[0],
            sim_label_func,
            hours_per_year
        )
    elif n_warming_levels == 1 and n_simulations > 1:

        return _create_single_wl_multi_sim_dataframe(
            profile_data,
            warming_levels[0],
            simulations,
            sim_label_func,
            hours_per_year
        )

    elif n_warming_levels > 1 and n_simulations == 1:
        return _create_multi_wl_single_sim_dataframe(
            profile_data,
            warming_levels,
            simulations[0],
            sim_label_func,
            hours_per_year
        )
    else:
        return _create_multi_wl_multi_sim_dataframe(
            profile_data,
            warming_levels,
            simulations,
            sim_label_func,
            hours_per_year
        )

In [6]:
def test_compute_profile(data: xr.DataArray, q=0.5) -> pd.DataFrame:
    """
    Calculates the standard year climate profile for warming level data using 8760
    analysis.

    This function handles global warming levels approach using time_delta coordinate.
    Processes all 30 years of warming level data centered around the year a warming level
    is reached, computes the specified quantile for each hour of the year across all years,
    then selects the actual data value closest to that quantile (not interpolated),
    and returns a characteristic profile of 8760 hours (one year) for each warming level
    and simulation combination.

    Parameters
    ----------
    data : xr.DataArray
        Hourly base-line subtracted data for one variable with warming_level,
        time_delta, and simulation dimensions. Expected to contain ~30 years
        (262,800 hours) of data for each warming level and simulation.

    days_in_year : int, optional
        Either 366 or 365, depending on whether or not the year is a leap year.
        Default to 365 days

    q : float, optional
        Quantile value for selecting representative values (0.0 to 1.0).
        Default is 0.5 (median).

    Returns
    -------
    pd.DataFrame
        Standard year table for each warming level and simulation,
        with days of year as the index and hour of day as the columns.
        Multi-index columns include Hour, Warming_Level, and Simulation dimensions.

    """
    # Check for simulation dimension
    has_simulation = "simulation" in data.dims
    if has_simulation:
        n_simulations = len(data.simulation)
        simulations = data.simulation.values
    else:
        n_simulations = 1
        simulations = [None]

    # Get all available time_delta data (all 30 years)
    hours_per_day = 24
    hours_per_year = 8760
    total_hours = len(data.time_delta)
    n_years = total_hours // hours_per_year

    print(f"      📊 Processing {total_hours:,} hours ({n_years} years) of data")
    print(f"      🎯 Computing {q*100:.0f}th percentile for each hour of year")

    # Create hour-of-year coordinate for all data (cycling through 1-8760)
    hour_of_year_all = np.tile(np.arange(1, hours_per_year + 1), n_years)[:total_hours]
    data = data.assign_coords(hour_of_year=("time_delta", hour_of_year_all))

    warming_levels = data.warming_level.values

    # Create helper function to extract meaningful simulation labels
    def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
        """Extract meaningful simulation label from simulation identifier."""
        if sim is None:
            return f"Sim_{sim_idx+1}"

        sim_str = str(sim)
        if "WRF_" in sim_str:
            # Parse simulation name format: WRF_GCM_params_scenario
            # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
            parts = sim_str.split("_")
            if len(parts) >= 4:
                gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
                params = parts[2]  # e.g., r11i1p1f1
                scenario = parts[3]  # e.g., historical+ssp245

                # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
                if "ssp" in scenario:
                    ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                    ssp = f"ssp{ssp_part}"
                else:
                    ssp = "hist"  # fallback for historical-only

                return f"{gcm}-{params}-{ssp}"
            elif len(parts) >= 2:
                # Fallback for shorter format
                return f"{parts[1]}-{sim_idx+1}"
            else:
                return f"Sim_{sim_idx+1}"
        else:
            # Ensure uniqueness by adding index for non-WRF format
            base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
            return f"{base_name}-{sim_idx+1}"

    # Process all data using quantile computation across years
    print(
        f"      ⚙️ Computing quantiles for {len(warming_levels)} warming level(s) and {n_simulations} simulation(s)"
    )

    # Eagerly compute all dask data at once (one round-trip to scheduler)
    if hasattr(data.data, "chunks"):
        print("      📥 Loading data into memory...")
        from dask.diagnostics import ProgressBar

        with ProgressBar():
            data = data.compute()

    # Initialize storage for profiles
    profile_data = {}

    # Progress tracking
    total_combinations = len(warming_levels) * n_simulations
    with tqdm(
        total=total_combinations,
        desc="      Computing profiles",
        unit="combo",
        leave=False,
    ) as pbar:

        for wl_idx, wl in enumerate(warming_levels):
            for sim_idx, sim in enumerate(simulations):
                # Get simulation label
                sim_label = _get_simulation_label(sim, sim_idx)

                # Select data for this warming level and simulation combination
                if has_simulation:
                    subset_data = data.isel(warming_level=wl_idx, simulation=sim_idx)
                else:
                    subset_data = data.isel(warming_level=wl_idx)

                # Vectorized quantile computation using numpy
                # Reshape raw values into (n_years, hours_per_year) then compute
                # the quantile across years for each hour-of-year position
                values = subset_data.values
                n_total = len(values)
                usable = (n_total // hours_per_year) * hours_per_year
                year_hour_matrix = values[:usable].reshape(-1, hours_per_year)

                # Compute quantile targets for each of the 8760 hour positions
                quantile_targets = np.nanquantile(
                    year_hour_matrix, q, axis=0
                )  # shape: (8760,)

                # For each hour position, find the actual year whose value is
                # closest to the quantile (avoids interpolation)
                diffs = np.abs(
                    year_hour_matrix - quantile_targets[np.newaxis, :]
                )  # (n_years, 8760)
                closest_year_idx = np.nanargmin(diffs, axis=0)  # (8760,)
                profile_1d = year_hour_matrix[
                    closest_year_idx, np.arange(hours_per_year)
                ]

                # Store the profile
                key = (f"WL_{wl}", sim_label)
                profile_data[key] = profile_1d

                pbar.update(1)

    df_profile = _construct_profile_dataframe(
        profile_data=profile_data,
        warming_levels=warming_levels,
        simulations=simulations,
        sim_label_func=_get_simulation_label,
        hours_per_year=hours_per_year
    )

    # # Determine which formatting function to use based on the structure
    # _format_based_on_structure(df_profile)

    # Prepare metadata dictionary
    metadata = {
        "quantile": q,
        "method": "8760 analysis - actual data closest to quantile across 30 years",
        "description": f"Climate profile computed using actual data values closest to the {q*100:.0f}th percentile of hourly data",
    }

    # Add original data attributes if available
    if hasattr(data, "attrs"):
        if "units" in data.attrs:
            metadata["units"] = data.attrs["units"]
        if "extended_description" in data.attrs:
            metadata["extended_description"] = data.attrs["extended_description"]
        if "variable_id" in data.attrs:
            metadata["variable_name"] = data.attrs["variable_id"]
        elif hasattr(data, "name") and data.name:
            metadata["variable_name"] = data.name

    # Set all metadata using the helper function
    set_profile_metadata(df_profile, metadata)

    print(f"      ✅ Profile computation complete! Final shape: {df_profile.shape}")
    print(
        f"         With index: {df_profile.index.name}, columns: {df_profile.columns.names}"
    )
    if hasattr(data, "attrs") and "units" in data.attrs:
        print(f"         Units: {data.attrs['units']}")

    return df_profile #profile_data_reshaped, profile_data_1d

In [7]:
test_simple = test_compute_profile(simple_data, q=0.5)
test_multi_sim = test_compute_profile(multi_sim_data, q=0.5)
test_multi_both = test_compute_profile(multi_both_data, q=0.5)
test_multi_wl = test_compute_profile(multi_wl_data, q=0.5)

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 1 warming level(s) and 1 simulation(s)


      Computing profiles:   0%|          | 0/1 [00:00<?, ?combo/s]

      ✅ Profile computation complete! Final shape: (8760, 1)
         With index: None, columns: [None]
         Units: degC
      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 1 warming level(s) and 2 simulation(s)


      Computing profiles:   0%|          | 0/2 [00:00<?, ?combo/s]

      ✅ Profile computation complete! Final shape: (8760, 2)
         With index: None, columns: [None]
         Units: degC
      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 2 warming level(s) and 2 simulation(s)


      Computing profiles:   0%|          | 0/4 [00:00<?, ?combo/s]

      ✅ Profile computation complete! Final shape: (8760, 4)
         With index: None, columns: ['Warming_Level', 'Simulation']
         Units: degC
      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 2 warming level(s) and 1 simulation(s)


      Computing profiles:   0%|          | 0/2 [00:00<?, ?combo/s]

      ✅ Profile computation complete! Final shape: (8760, 2)
         With index: None, columns: ['Warming_Level', 'Simulation']
         Units: degC


### Original 

In [8]:
from climakitae.explore.og_standard_year_profile import (
    get_climate_profile,
    export_profile_to_csv,
    set_profile_metadata,
    compute_profile,
    _format_based_on_structure,
    _construct_profile_dataframe,
    _create_simple_dataframe,
    _create_single_wl_multi_sim_dataframe,
    _create_multi_wl_single_sim_dataframe,
    _create_multi_wl_multi_sim_dataframe,
    _stack_profile_data,
)

In [9]:
og_test_simple = compute_profile(simple_data, days_in_year=365, q=0.5)
og_test_multi_sim = compute_profile(multi_sim_data, days_in_year=365, q=0.5)
og_test_multi_both = compute_profile(multi_both_data, days_in_year=365, q=0.5)
og_test_multi_wl = compute_profile(multi_wl_data, days_in_year=365, q=0.5)

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 1 warming level(s) and 1 simulation(s)


      Computing profiles:   0%|          | 0/1 [00:00<?, ?combo/s]

      ✅ Profile computation complete! Final shape: (365, 24)
         With index: Day of Year, columns: ['Hour', 'Simulation']
         Units: degC
      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 1 warming level(s) and 2 simulation(s)


      Computing profiles:   0%|          | 0/2 [00:00<?, ?combo/s]

      ✅ Profile computation complete! Final shape: (365, 48)
         With index: Day of Year, columns: ['Hour', 'Simulation']
         Units: degC
      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 2 warming level(s) and 2 simulation(s)


      Computing profiles:   0%|          | 0/4 [00:00<?, ?combo/s]

      ✅ Profile computation complete! Final shape: (365, 96)
         With index: Day of Year, columns: ['Hour', 'Warming_Level', 'Simulation']
         Units: degC
      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 2 warming level(s) and 1 simulation(s)


      Computing profiles:   0%|          | 0/2 [00:00<?, ?combo/s]

      ✅ Profile computation complete! Final shape: (365, 48)
         With index: Day of Year, columns: ['Hour', 'Warming_Level']
         Units: degC


### Export

In [ ]:
warming_levels = [1.5]
warming_levels_2 = [1.5, 2.0]
simulations = ["sim1"]
simulations_2 = ["sim1", "sim2"]

In [ ]:
profile_selections = {
    ## Required variable and profile arguments
    "variable": "Air Temperature at 2m",
    "resolution": "3 km",
    "q": 0.5,
    "units": "degF",
    ## Required approach arguments, Options: "Warming Level", "Time"
    "approach": "Warming Level",
    "warming_level": [1.2], # GWL-option only
    # "centered_year": centered_year, # Time-based option only
    ## Required location arguments -- Uncomment your desired location option and modify
    "stations": ["Sacramento Executive Airport (KSAC)"],
    # "latitude": (latitude - 0.02, latitude + 0.02),
    # "longitude": (longitude - 0.02, longitude + 0.02),
    # "cached_area": area_name,
    ## Additional optional arguments -- Uncomment any desired options and modify
    # "no_delta": True,
    # "warming_level_window": 10,
    # "time_profile_scenario": "SSP2-4.5,
    # "bias_adjusted_models": True,
}

#### Cascading reformat

In [18]:
from climakitae.explore.standard_year_profile import (
    read_csv_file,
    _check_cached_area,
    _check_lat_lon,
    _check_stations,
    _get_clean_standardyr_filename,
    _compute_difference_profile,
    _find_matching_historic_column,
    _get_historic_hour_mean
)
from typing import Tuple
from typing import Any, Dict
from climakitae.core.paths import HADISD_STATIONS_URL, VARIABLE_DESCRIPTIONS_CSV_PATH

test_simple
test_multi_sim
test_multi_both
test_multi_wl

og_test_simple
og_test_multi_sim
og_test_multi_both
og_test_multi_wl

In [ ]:
test_multi_sim

In [ ]:
# _compute_difference_profile()
# OG case

print(isinstance(og_test_simple.columns, pd.MultiIndex))
print(isinstance(og_test_multi_sim.columns, pd.MultiIndex))
print(isinstance(og_test_multi_both.columns, pd.MultiIndex))
print(isinstance(og_test_multi_wl.columns, pd.MultiIndex))
# => _compute_multiindex_difference(future_profile, historic_profile) used


print(isinstance(test_simple.columns, pd.MultiIndex))
print(isinstance(test_multi_sim.columns, pd.MultiIndex))
print(isinstance(test_multi_both.columns, pd.MultiIndex))
print(isinstance(test_multi_wl.columns, pd.MultiIndex))


In [ ]:
def _compute_simulation_paired_difference(
    future_profile: pd.DataFrame,
    historic_profile: pd.DataFrame,
    future_levels: list,
    historic_levels: list,
) -> pd.DataFrame:
    """
    Compute difference for profiles with matching simulations.

    Parameters
    ----------
    future_profile : pd.DataFrame
        Future profile with Simulation level
    historic_profile : pd.DataFrame
        Historic profile with Simulation level
    future_levels : list
        Names of future profile column levels
    historic_levels : list
        Names of historic profile column levels

    Returns
    -------
    pd.DataFrame
        Difference profile with paired simulations
    """
    # Check for duplicate columns and handle them
    if not future_profile.columns.is_unique:
        print(
            "   ⚠️  Warning: Found duplicate columns in future profile. Removing duplicates."
        )
        future_profile = future_profile.loc[:, ~future_profile.columns.duplicated()]

    if not historic_profile.columns.is_unique:
        print(
            "   ⚠️  Warning: Found duplicate columns in historic profile. Removing duplicates."
        )
        historic_profile = historic_profile.loc[
            :, ~historic_profile.columns.duplicated()
        ]

    difference_profile = future_profile.copy()

    # Get unique simulations from both profiles
    future_sims = future_profile.columns.get_level_values("Simulation").unique()
    historic_sims = historic_profile.columns.get_level_values("Simulation").unique()

    # Find common simulations
    common_sims = set(future_sims) & set(historic_sims)

    if not common_sims:
        print(
            "   ⚠️  Warning: No matching simulations found between future and historic profiles!"
        )
        print(f"      Future simulations: {list(future_sims)}")
        print(f"      Historic simulations: {list(historic_sims)}")
        # Fall back to using mean of historic
        # Note: axis parameter removed in pandas 2.2, use level-based groupby instead
        historic_mean = historic_profile.T.groupby(level="Hour").mean().T
        for col in future_profile.columns:
            hour = col[0] if "Hour" in future_levels else col[-1]
            difference_profile.loc[:, col] = future_profile[col] - historic_mean[hour]
    else:
        # Compute differences for matching simulations
        n_cols = len(future_profile.columns)
        with tqdm(
            total=n_cols, desc="   Computing paired differences", unit="column"
        ) as pbar:
            for col in future_profile.columns:
                historic_col = _find_matching_historic_column(
                    col, future_levels, historic_profile, historic_levels
                )
                if historic_col and historic_col in historic_profile.columns:
                    difference_profile.loc[:, col] = (
                        future_profile[col] - historic_profile[historic_col]
                    )
                else:
                    # Use mean of historic for that hour
                    hour = col[0]  # Assuming hour is first level
                    historic_hour_mean = _get_historic_hour_mean(
                        historic_profile, historic_levels, hour
                    )
                    difference_profile.loc[:, col] = (
                        future_profile[col] - historic_hour_mean
                    )
                pbar.update(1)

    return difference_profile


def _compute_warming_level_difference(
    future_profile: pd.DataFrame,
    historic_profile: pd.DataFrame,
    future_levels: list,
    historic_levels: list,
) -> pd.DataFrame:
    """
    Compute difference for profiles with warming levels but no simulations.

    Parameters
    ----------
    future_profile : pd.DataFrame
        Future profile with Warming_Level
    historic_profile : pd.DataFrame
        Historic profile
    future_levels : list
        Names of future profile column levels
    historic_levels : list
        Names of historic profile column levels

    Returns
    -------
    pd.DataFrame
        Difference profile
    """
    # Check for duplicate columns and handle them
    if not future_profile.columns.is_unique:
        print(
            "   ⚠️  Warning: Found duplicate columns in future profile. Removing duplicates."
        )
        future_profile = future_profile.loc[:, ~future_profile.columns.duplicated()]

    if not historic_profile.columns.is_unique:
        print(
            "   ⚠️  Warning: Found duplicate columns in historic profile. Removing duplicates."
        )
        historic_profile = historic_profile.loc[
            :, ~historic_profile.columns.duplicated()
        ]

    difference_profile = future_profile.copy()

    n_cols = len(future_profile.columns)
    with tqdm(total=n_cols, desc="   Computing differences", unit="column") as pbar:
        for col in future_profile.columns:
            # Use future_levels to determine which position contains the hour
            if "Hour" in future_levels:
                hour_idx = future_levels.index("Hour")
                hour = col[hour_idx]
            else:
                # Fallback: assume first position is hour
                hour = col[0]

            if hour in historic_profile.columns:
                difference_profile.loc[:, col] = (
                    future_profile[col] - historic_profile[hour]
                )
            else:
                # Try to find corresponding hour in historic MultiIndex
                if historic_levels and "Hour" in historic_levels:
                    try:
                        historic_hour = historic_profile.xs(
                            hour, level="Hour", axis=1
                        ).iloc[:, 0]
                    except (KeyError, IndexError):
                        # If xs fails, fall back to first column
                        historic_hour = historic_profile.iloc[:, 0]
                else:
                    historic_hour = historic_profile.iloc[
                        :, 0
                    ]  # Fall back to first column
                difference_profile.loc[:, col] = future_profile[col] - historic_hour
            pbar.update(1)

    return difference_profile


def _compute_mixed_index_difference(
    future_profile: pd.DataFrame, historic_profile: pd.DataFrame
) -> pd.DataFrame:
    """
    Compute difference when future has MultiIndex and historic has simple index.

    Parameters
    ----------
    future_profile : pd.DataFrame
        Future profile with MultiIndex columns
    historic_profile : pd.DataFrame
        Historic profile with simple columns

    Returns
    -------
    pd.DataFrame
        Difference profile
    """
    # Check for duplicate columns and handle them
    if not future_profile.columns.is_unique:
        print(
            "   ⚠️  Warning: Found duplicate columns in future profile. Removing duplicates."
        )
        future_profile = future_profile.loc[:, ~future_profile.columns.duplicated()]

    if not historic_profile.columns.is_unique:
        print(
            "   ⚠️  Warning: Found duplicate columns in historic profile. Removing duplicates."
        )
        historic_profile = historic_profile.loc[
            :, ~historic_profile.columns.duplicated()
        ]

    difference_profile = future_profile.copy()

    n_cols = len(future_profile.columns)
    with tqdm(total=n_cols, desc="   Computing differences", unit="column") as pbar:
        for col in future_profile.columns:
            historic_value = _find_matching_historic_value(
                col, future_profile, historic_profile
            )
            difference_profile.loc[:, col] = future_profile[col] - historic_value
            pbar.update(1)

    return difference_profile


def _compute_simple_difference(
    future_profile: pd.DataFrame, historic_profile: pd.DataFrame
) -> pd.DataFrame:
    """
    Compute difference for profiles with simple (non-MultiIndex) columns.

    Parameters
    ----------
    future_profile : pd.DataFrame
        Future profile
    historic_profile : pd.DataFrame
        Historic profile

    Returns
    -------
    pd.DataFrame
        Difference profile
    """
    if list(future_profile.columns) == list(historic_profile.columns):
        print("   ✓ Columns match - computing element-wise difference")
        return future_profile - historic_profile
    else:
        print("   ⚠️  Warning: Column mismatch between future and historic profiles")
        print(f"      Future columns: {list(future_profile.columns)[:5]}...")
        print(f"      Historic columns: {list(historic_profile.columns)[:5]}...")
        # Try to align by position
        min_cols = min(len(future_profile.columns), len(historic_profile.columns))
        return future_profile.iloc[:, :min_cols] - historic_profile.iloc[:, :min_cols]



explode _compute_multiindex_difference

In [ ]:
future_profile = og_test_simple
historic_profile = og_test_multi_sim

In [ ]:
future_levels = future_profile.columns.names
historic_levels = historic_profile.columns.names
print(future_levels)
print(historic_levels)

In [ ]:
if "Simulation" in future_levels and "Simulation" in historic_levels:
    print("if")
    result = _compute_simulation_paired_difference(
        future_profile, historic_profile, future_levels, historic_levels
    )
elif "Warming_Level" in future_levels and "Simulation" not in future_levels:
    print("elif")
    result = _compute_warming_level_difference(
        future_profile, historic_profile, future_levels, historic_levels
    )
else:
    # Default to simple difference if structure is unexpected
    print("else")
    result = _compute_simple_difference(future_profile, historic_profile)

In [ ]:
# _compute_simulation_paired_difference()



In [13]:
og_test_multi_both

Hour                 1                                       2             \
Warming_Level    WL_1.5              WL_2.0              WL_1.5             
Simulation       sim1-1    sim2-2    sim1-1    sim2-2    sim1-1    sim2-2   
Day of Year                                                                 
Jan-01         0.948229  0.928774  0.944693  0.916198  0.437616  0.585622   
Jan-02         0.313472  0.109887  0.130531  0.696905  0.680743  0.742051   
Jan-03         0.240417  0.771817  0.986100  0.853404  0.323239  0.863257   
Jan-04         0.407329  0.362840  0.563975  0.636664  0.029675  0.244741   
Jan-05         0.117492  0.348824  0.209363  0.177947  0.420604  0.312531   
...                 ...       ...       ...       ...       ...       ...   
Dec-27         0.764297  0.966155  0.190062  0.435675  0.824999  0.237863   
Dec-28         0.237602  0.751390  0.148686  0.480256  0.259982  0.671027   
Dec-29         0.657983  0.657435  0.124243  0.619536  0.599134  0.099797   
Dec-30         0.101466  0.370722  0.571459  0.528704  0.943925  0.632167   
Dec-31         0.138485  0.909667  0.403476  0.600781  0.665120  0.480448   

Hour                                     3             ...        22  \
Warming_Level    WL_2.0              WL_1.5            ...    WL_2.0   
Simulation       sim1-1    sim2-2    sim1-1    sim2-2  ...    sim1-1   
Day of Year                                            ...             
Jan-01         0.655483  0.112290  0.278207  0.244311  ...  0.223408   
Jan-02         0.382413  0.176321  0.147276  0.473265  ...  0.172276   
Jan-03         0.080863  0.673588  0.228200  0.116813  ...  0.631734   
Jan-04         0.085941  0.542209  0.474389  0.275216  ...  0.723251   
Jan-05         0.141806  0.913071  0.581132  0.422144  ...  0.598061   
...                 ...       ...       ...       ...  ...       ...   
Dec-27         0.510084  0.181175  0.986717  0.852278  ...  0.111180   
Dec-28         0.459804  0.083408  0.717199  0.438323  ...  0.612543   
Dec-29         0.389744  0.657900  0.479971  0.967799  ...  0.679685   
Dec-30         0.251184  0.394726  0.404748  0.680472  ...  0.508011   
Dec-31         0.293676  0.744141  0.962034  0.126558  ...  0.243590   

Hour                           23                                      24  \
Warming_Level              WL_1.5              WL_2.0              WL_1.5   
Simulation       sim2-2    sim1-1    sim2-2    sim1-1    sim2-2    sim1-1   
Day of Year                                                                 
Jan-01         0.661059  0.070262  0.818975  0.736766  0.393406  0.111364   
Jan-02         0.587799  0.939839  0.528130  0.505160  0.638708  0.168941   
Jan-03         0.971704  0.944654  0.782398  0.106289  0.302719  0.098535   
Jan-04         0.577691  0.535907  0.942612  0.566766  0.099907  0.122743   
Jan-05         0.738275  0.301644  0.070766  0.756875  0.575819  0.519173   
...                 ...       ...       ...       ...       ...       ...   
Dec-27         0.990915  0.091657  0.859616  0.324312  0.084291  0.136360   
Dec-28         0.821111  0.176480  0.349616  0.405004  0.467215  0.693478   
Dec-29         0.950480  0.617950  0.896023  0.179199  0.984761  0.682379   
Dec-30         0.876530  0.080680  0.666141  0.627720  0.086366  0.340496   
Dec-31         0.414718  0.308932  0.083385  0.159690  0.088954  0.548927   

Hour                                         
Warming_Level              WL_2.0            
Simulation       sim2-2    sim1-1    sim2-2  
Day of Year                                  
Jan-01         0.766710  0.515123  0.688135  
Jan-02         0.274702  0.010027  0.432452  
Jan-03         0.722814  0.372765  0.376888  
Jan-04         0.990390  0.997690  0.670769  
Jan-05         0.449088  0.274561  0.485944  
...                 ...       ...       ...  
Dec-27         0.901197  0.652949  0.873436  
Dec-28         0.495285  0.073020  0.682123  
Dec-29         0.885187  0.277424  0.925113  
Dec-30   

In [ ]:
def _compute_multiindex_difference(
    future_profile: pd.DataFrame, historic_profile: pd.DataFrame
) -> pd.DataFrame:
    """
    Compute difference when both profiles have MultiIndex columns.

    Parameters
    ----------
    future_profile : pd.DataFrame
        Future profile with MultiIndex columns
    historic_profile : pd.DataFrame
        Historic profile with MultiIndex columns

    Returns
    -------
    pd.DataFrame
        Difference profile
    """
    future_levels = future_profile.columns.names
    historic_levels = historic_profile.columns.names

    if "Simulation" in future_levels and "Simulation" in historic_levels:
        return _compute_simulation_paired_difference(
            future_profile, historic_profile, future_levels, historic_levels
        )
    elif "Warming_Level" in future_levels and "Simulation" not in future_levels:
        return _compute_warming_level_difference(
            future_profile, historic_profile, future_levels, historic_levels
        )
    else:
        # Default to simple difference if structure is unexpected
        return _compute_simple_difference(future_profile, historic_profile)

In [ ]:
difference_profile = _compute_difference_profile(test_multi_sim, historic_profile = None)

exploding _compute_simulation_paired_difference()

In [83]:
# future_profile = og_test_simple
# historic_profile = og_test_multi_sim

future_profile = test_simple
historic_profile = test_multi_sim

future_levels = future_profile.columns.names
historic_levels = historic_profile.columns.names

In [92]:
# Compute differences for matching simulations
n_cols = len(future_profile.columns)

In [106]:
col = "sim1"

In [108]:
difference_profile.loc[:, "sim1-1"] = (
    future_profile["sim1-1"] - historic_profile["sim1"]
)

In [ ]:
if col in historic_profile.columns:
    difference_profile.loc[:, col] = (
        future_profile[col] - historic_profile[col]
    )
else:
    # Use mean of historic for that hour
    hour = col[0]  # Assuming hour is first level
    historic_mean = test_multi_sim.T.mean()
    difference_profile.loc[:, col] = (
        future_profile[col] - historic_mean
    )

In [ ]:
# Check for duplicate columns and handle them
if not future_profile.columns.is_unique:
    print(
        "   ⚠️  Warning: Found duplicate columns in future profile. Removing duplicates."
    )
    future_profile = future_profile.loc[:, ~future_profile.columns.duplicated()]

if not historic_profile.columns.is_unique:
    print(
        "   ⚠️  Warning: Found duplicate columns in historic profile. Removing duplicates."
    )
    historic_profile = historic_profile.loc[:, ~historic_profile.columns.duplicated()]


difference_profile = future_profile.copy()

# Get unique simulations from both profiles
future_sims = future_profile.columns.unique()
historic_sims = historic_profile.columns.unique()

# Find common simulations
common_sims = set(future_sims) & set(historic_sims)

historic_mean = historic_profile.T.mean()

if not common_sims:
    print(
        "   ⚠️  Warning: No matching simulations found between future and historic profiles!"
    )
    print(f"      Future simulations: {list(future_sims)}")
    print(f"      Historic simulations: {list(historic_sims)}")
    # Fall back to using mean of historic
    # Note: axis parameter removed in pandas 2.2, use level-based groupby instead
    for col in future_profile.columns:
        difference_profile.loc[:, col] = future_profile[col] - historic_mean
else:
    # Compute differences for matching simulations
    n_cols = len(future_profile.columns)
    with tqdm(
        total=n_cols, desc="   Computing paired differences", unit="column"
    ) as pbar:
        for col in future_profile.columns:
            if col in historic_profile.columns:
                difference_profile.loc[:, col] = (
                    future_profile[col] - historic_profile[col]
                )
            else:
                difference_profile.loc[:, col] = (
                    future_profile[col] - historic_mean
                )
            pbar.update(1)

   ⚠️  Warning: No matching simulations found between future and historic profiles!
      Future simulations: ['sim1-1']
      Historic simulations: ['sim1', 'sim2']


ValueError: level name Hour is not the name of the index

## Testing

In [ ]:
# Set up the Standard Year generator
profile_selections = {  
    ## Required variable and profile arguments
    "variable": "Air Temperature at 2m",
    "resolution": "3 km",
    "q": 0.5,
    "units": "degF",

    ## Required approach arguments, Options: "Warming Level", "Time"
    "approach": "Warming Level",
    # "warming_level": [warming_level], # GWL-option only
    # "centered_year": centered_year, # Time-based option only
    
    ## Required location arguments -- Uncomment your desired location option and modify
    "stations": ["Sacramento Executive Airport (KSAC)"], 
    # "latitude": (latitude - 0.02, latitude + 0.02), 
    # "longitude": (longitude - 0.02, longitude + 0.02),  
    # "cached_area": area_name, 

    ## Additional optional arguments -- Uncomment any desired options and modify
    # "no_delta": True, 
    # "warming_level_window": 10,
    # "time_profile_scenario": "SSP2-4.5,
    # "bias_adjusted_models": True,
}

# Generate the climate profile
profile = get_climate_profile(**profile_selections)

In [ ]:
# the function uses the previously defined profile selections to generate the output file name
export_profile_to_csv(profile, **profile_selections)